# Checkpoint 4 — Mock Contest 4 Solutions

Reference solutions for all 7 problems. Each is a complete stdin/stdout program; the cells below are display-only mirrors of the solver files in the `assets/` folder.

## Question 1

**Notice:** flood-fill each pasture exactly once before reading any question. Scan the map, and at every grass cell without a label, start a new pasture number and let the recursive `fill` label every grass cell connected up, down, left, or right, adding one to that pasture's size for each cell. Then a question is a single lookup: `0` for a fence, otherwise the size of the pasture whose label the cell carries. The fills touch each of at most `400` cells once, and each of the `200000` questions takes constant time, where refilling for every question would repeat the work up to `200000` times.

In [ ]:
import sys

data = sys.stdin.read()
tokens = data.split()
rows = int(tokens[0])
cols = int(tokens[1])
queries = int(tokens[2])
grid = []
label = []
r = 0
while r < rows:
    grid.append(tokens[3 + r])
    label_row = []
    c = 0
    while c < cols:
        label_row.append(-1)
        c = c + 1
    label.append(label_row)
    r = r + 1


def fill(cr, cc, pasture, sizes):
    label[cr][cc] = pasture
    sizes[pasture] = sizes[pasture] + 1
    neighbours = [(cr - 1, cc), (cr + 1, cc), (cr, cc - 1), (cr, cc + 1)]
    for spot in neighbours:
        nr = spot[0]
        nc = spot[1]
        if 0 <= nr and nr < rows and 0 <= nc and nc < cols:
            if grid[nr][nc] == "." and label[nr][nc] == -1:
                fill(nr, nc, pasture, sizes)


sizes = []
r = 0
while r < rows:
    c = 0
    while c < cols:
        if grid[r][c] == "." and label[r][c] == -1:
            sizes.append(0)
            fill(r, c, len(sizes) - 1, sizes)
        c = c + 1
    r = r + 1

answers = []
i = 0
while i < queries:
    qr = int(tokens[3 + rows + 2 * i])
    qc = int(tokens[4 + rows + 2 * i])
    if grid[qr][qc] == "#":
        answers.append("0")
    else:
        answers.append(str(sizes[label[qr][qc]]))
    i = i + 1
for answer in answers:
    print(answer)


## Question 2

**Notice:** multi-source BFS: put every `M` cell on the `deque` with time `0` before the first `popleft`, so the mold spreads from all patches at once. Each clean cell is first reached at the minute it becomes moldy, and the queue hands cells out in order of time, so the largest time taken from the queue is the answer. Count the clean cells at the start and the cells the BFS reaches; if some were never reached, print `-1`. With no clean cell the answer stays `0`. Every cell enters the queue at most once, so the BFS takes one pass over at most `200000` cells, where re-scanning the map every minute could take `200000` scans.

In [ ]:
from collections import deque
import sys

data = sys.stdin.read()
tokens = data.split()
rows = int(tokens[0])
cols = int(tokens[1])
grid = []
r = 0
while r < rows:
    grid.append(tokens[2 + r])
    r = r + 1

queue = deque()
visited = set()
clean = 0
r = 0
while r < rows:
    c = 0
    while c < cols:
        if grid[r][c] == "M":
            queue.append((r, c, 0))
            visited.add((r, c))
        elif grid[r][c] == ".":
            clean = clean + 1
        c = c + 1
    r = r + 1

reached = 0
minutes = 0
while len(queue) > 0:
    item = queue.popleft()
    cr = item[0]
    cc = item[1]
    time = item[2]
    if time > minutes:
        minutes = time
    neighbours = [(cr - 1, cc), (cr + 1, cc), (cr, cc - 1), (cr, cc + 1)]
    for spot in neighbours:
        nr = spot[0]
        nc = spot[1]
        if 0 <= nr and nr < rows and 0 <= nc and nc < cols:
            if grid[nr][nc] == "." and (nr, nc) not in visited:
                visited.add((nr, nc))
                reached = reached + 1
                queue.append((nr, nc, time + 1))
if reached < clean:
    print("-1")
else:
    print(str(minutes))


## Question 3

**Notice:** store the rivals in an adjacency-list dictionary. `place(student, side)` puts a student on team `1` or `2` and then places each unplaced rival on the other team (`3 - side`); if a rival already sits on the same team, the split is impossible and `False` travels back up the calls. Rivalries may form several separate groups, so start a new `place` from every student who is still unplaced. Each student is placed once and each rivalry is checked from both ends, so `500` students and `20000` rivalries take one pass, where trying every split would mean `2^500` choices.

In [ ]:
import sys

data = sys.stdin.read()
tokens = data.split()
n = int(tokens[0])
m = int(tokens[1])
adj = {}
node = 1
while node <= n:
    adj[node] = []
    node = node + 1
i = 0
while i < m:
    u = int(tokens[2 + 2 * i])
    v = int(tokens[3 + 2 * i])
    adj[u].append(v)
    adj[v].append(u)
    i = i + 1

team = {}


def place(student, side):
    team[student] = side
    for rival in adj[student]:
        if rival not in team:
            if not place(rival, 3 - side):
                return False
        elif team[rival] == side:
            return False
    return True


possible = True
node = 1
while node <= n and possible:
    if node not in team:
        possible = place(node, 1)
    node = node + 1
if possible:
    print("YES")
else:
    print("NO")


## Question 4

**Notice:** converging two pointers: record `min(h[lo], h[hi]) * (hi - lo)`, then move the pointer at the SHORTER post inward (move `hi` on a tie). Every trough that keeps the shorter post and uses a closer partner is no taller and narrower, so it cannot hold more water; dropping that post loses nothing. Each step moves one pointer, so the scan takes `N - 1` steps instead of checking all `N^2 / 2` pairs.

In [ ]:
import sys

data = sys.stdin.read()
tokens = data.split()
n = int(tokens[0])
heights = []
i = 0
while i < n:
    heights.append(int(tokens[1 + i]))
    i = i + 1
lo = 0
hi = n - 1
best = 0
while lo < hi:
    water = min(heights[lo], heights[hi]) * (hi - lo)
    if water > best:
        best = water
    if heights[lo] < heights[hi]:
        lo = lo + 1
    else:
        hi = hi - 1
print(str(best))


## Question 5

**Notice:** first count the different flavors in the whole row, `needed`. Then slide a window: add `flavors[right]`, raising `covered` when its count goes from `0` to `1`. While the window holds all `needed` flavors, record its length and move `left` forward, lowering `covered` when a flavor's count drops to `0`. Every stretch that holds every flavor is measured at its shortest for each right end, so the smallest length recorded is the answer. Each tub enters and leaves the window at most once, so `200000` tubs take a linear scan.

In [ ]:
import sys

data = sys.stdin.read()
tokens = data.split()
n = int(tokens[0])
flavors = []
kinds = set()
i = 0
while i < n:
    flavor = int(tokens[1 + i])
    flavors.append(flavor)
    kinds.add(flavor)
    i = i + 1
needed = len(kinds)

counts = {}
covered = 0
best = n
left = 0
right = 0
while right < n:
    flavor = flavors[right]
    if flavor not in counts:
        counts[flavor] = 0
    counts[flavor] = counts[flavor] + 1
    if counts[flavor] == 1:
        covered = covered + 1
    while covered == needed:
        span = right - left + 1
        if span < best:
            best = span
        leaving = flavors[left]
        counts[leaving] = counts[leaving] - 1
        if counts[leaving] == 0:
            covered = covered - 1
        left = left + 1
    right = right + 1
print(str(best))


## Question 6

**Notice:** keep `power = A^e` and `total = 1 + A + ... + A^(e - 1)` modulo `M` for the days handled so far, starting from `e = 0` with `power = 1 % M` and `total = 0`. Doubling `e` gives the old total plus a copy shifted by `A^e`, so `total` becomes `total + power * total` and `power` becomes `power * power`. Adding one day adds `power` to `total` and multiplies `power` by `A`. Read the bits of `E` from the most significant down: double for every bit, then add a day when the bit is `1`, reducing modulo `M` after every multiplication. `E = 0` has no bits and prints `0`, and `1 % M` makes `M = 1` print `0`. `10^18` has about `60` bits, so the program does a few hundred multiplications on numbers below `10^36`.

In [ ]:
import sys

data = sys.stdin.read()
tokens = data.split()
base = int(tokens[0])
days = int(tokens[1])
modulus = int(tokens[2])

bits = []
rest = days
while rest > 0:
    bits.append(rest % 2)
    rest = rest // 2

power = 1 % modulus
total = 0
factor = base % modulus
i = len(bits) - 1
while i >= 0:
    total = (total + power * total) % modulus
    power = power * power % modulus
    if bits[i] == 1:
        total = (total + power) % modulus
        power = power * factor % modulus
    i = i - 1
print(str(total))


## Question 7

**Notice:** BFS from `start` over a DIRECTED adjacency list (each edge added one way only), printing the visit order.

In [ ]:
from collections import deque
import sys

data = sys.stdin.read()
lines = data.split("\n")
header = lines[0].split()
n = int(header[0])
m = int(header[1])
start = int(header[2])
adj = {}
node = 1
while node <= n:
    adj[node] = []
    node = node + 1
i = 0
while i < m:
    parts = lines[1 + i].split()
    u = int(parts[0])
    v = int(parts[1])
    adj[u].append(v)
    i = i + 1
visited = {start}
queue = deque()
queue.append(start)
order = ""
while len(queue) > 0:
    u = queue.popleft()
    if len(order) > 0:
        order = order + " "
    order = order + str(u)
    for nb in adj[u]:
        if nb not in visited:
            visited.add(nb)
            queue.append(nb)
print(order)
